<a href="https://colab.research.google.com/github/Se886372/Starbucks-Order-Interest-Over-Time/blob/main/Starbucks_Timeseries_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## What This Code Does

This code grabs a data file (originally an Apple Numbers spreadsheet) from a GitHub repository online, downloads it into the notebook, and then opens it up to show what's inside — listing each sheet and table along with how many rows and columns it has, so you can see the file's structure before doing anything else with the data.


In [1]:
# Install the numbers-parser library
!pip install numbers-parser

import requests
from numbers_parser import Document
import pandas as pd

# Raw GitHub URL for your file
url = "https://raw.githubusercontent.com/Se886372/Starbucks-Order-Interest-Over-Time/main/time_series_US_20031231-1900_20260710-1049.numbers"

# Download it into the Colab environment
file_path = "/content/time_series_US_20031231-1900_20260710-1049.numbers"
response = requests.get(url)
response.raise_for_status()  # errors out loudly if the fetch fails (e.g. 404)

with open(file_path, "wb") as f:
    f.write(response.content)

print(f"Downloaded {len(response.content):,} bytes to {file_path}")

# Load the document
doc = Document(file_path)

# A .numbers file can contain multiple sheets, each with multiple tables
# List what's available first so you can pick the right one
for sheet in doc.sheets:
    print(f"Sheet: {sheet.name}")
    for table in sheet.tables:
        print(f"  Table: {table.name} — {table.num_rows} rows x {table.num_cols} cols")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.3/325.3 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 10.6 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.35.1 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.35.1 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.35.1 which is incompatible.


Downloaded 249,887 bytes to /content/time_series_US_20031231-1900_20260710-1049.numbers
Sheet: Sheet 1
  Table: time_series_US_20031231-1900_20260710-1049 — 272 rows x 9 cols


In [2]:
# Grab the first sheet and first table (adjust if your data lives elsewhere)
sheet = doc.sheets[0]
table = sheet.tables[0]

# Pull all rows, use the first row as the header
data = table.rows(values_only=True)
df = pd.DataFrame(data[1:], columns=data[0])

# Quick check
print(df.shape)
print(df.columns.tolist())
df.head(10)

(271, 9)
['Time', 'Pumpkin spice latte', 'Chai Latte', 'pink drink', 'Caramel Macchiato', 'nitro cold brew coffee', 'refresher', 'matcha tea', 'Frappuccino']


,Time,Pumpkin spice latte,Chai Latte,pink drink,Caramel Macchiato,nitro cold brew coffee,refresher,matcha tea,Frappuccino
0,2004-01-01,0.0,4.0,0.0,0.0,0.0,12.0,2.0,0.0
1,2004-02-01,0.0,2.0,0.0,0.0,0.0,10.0,0.0,0.0
2,2004-03-01,0.0,3.0,0.0,0.0,0.0,9.0,0.0,0.0
3,2004-04-01,0.0,2.0,0.0,0.0,0.0,10.0,0.0,0.0
4,2004-05-01,0.0,2.0,0.0,0.0,0.0,10.0,0.0,0.0
5,2004-06-01,0.0,1.0,2.0,0.0,0.0,11.0,0.0,0.0
6,2004-07-01,0.0,2.0,0.0,0.0,0.0,12.0,0.0,0.0
7,2004-08-01,0.0,2.0,1.0,0.0,0.0,10.0,0.0,0.0
8,2004-09-01,0.0,1.0,1.0,0.0,0.0,10.0,0.0,0.0
9,2004-10-01,2.0,2.0,1.0,0.0,0.0,9.0,0.0,0.0


In [27]:
import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, display
import json

# --- Date column (already confirmed as "Time") ---
date_col = "Time"
df[date_col] = pd.to_datetime(df[date_col])
df = df.sort_values(date_col)

value_cols = ['Pumpkin spice latte', 'Chai Latte', 'pink drink', 'Caramel Macchiato',
              'nitro cold brew coffee', 'refresher', 'matcha tea', 'Frappuccino']

colors = [
    '#D2691E',  # Pumpkin spice latte - burnt pumpkin orange
    '#A0522D',  # Chai Latte - cinnamon sienna
    '#FF66B2',  # pink drink - vibrant pink
    '#C68E17',  # Caramel Macchiato - caramel gold
    '#5DADE2',  # nitro cold brew coffee - light blue (iced/cold)
    '#E63950',  # refresher - berry red
    '#6B8E23',  # matcha tea - matcha olive-green
    '#8B5A2B',  # Frappuccino - mocha brown
]

# --- Build the figure with all 8 traces ---
fig = go.Figure()
for i, col in enumerate(value_cols):
    fig.add_trace(go.Scatter(
        x=df[date_col],
        y=df[col],
        mode='lines',
        name=col,
        line=dict(color=colors[i], width=3.0),
        opacity=0.9,
        hovertemplate='<b>%{x|%b %d, %Y}</b><br>' + col + ': %{y}<extra></extra>'
    ))

fig.update_layout(
    title=dict(
        text="Starbucks Orders Search Interest Over Time",
        font=dict(size=22, family="Arial", color="#1E3932"),
        x=0.02
    ),
    xaxis=dict(
        title="Date",
        showgrid=False,
        rangeslider=dict(visible=True),
        rangeselector=dict(
            buttons=list([
                dict(count=1, label="1M", step="month", stepmode="backward"),
                dict(count=6, label="6M", step="month", stepmode="backward"),
                dict(count=1, label="1Y", step="year", stepmode="backward"),
                dict(count=5, label="5Y", step="year", stepmode="backward"),
                dict(step="all", label="All")
            ]),
            bgcolor="#F0F0F0"
        )
    ),
    yaxis=dict(
        title="Search Interest",
        showgrid=True,
        gridcolor='rgba(0,0,0,0.06)'
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family="Arial", size=16, color="#333333"),
    hovermode='x unified',
    showlegend=False,  # replaced by the checkbox panel
    width=1700,
    height=800,
    margin=dict(l=60, r=20, t=110, b=60)
)

# --- Convert the chart to a raw HTML div (self-contained, no ipywidgets) ---
chart_div_id = "starbucks_chart_div"
chart_html = fig.to_html(
    include_plotlyjs='cdn',
    full_html=False,
    div_id=chart_div_id
)

# --- Build the checkbox panel as plain HTML + JS ---
checkbox_rows = ""
for i, col in enumerate(value_cols):
    checkbox_rows += f"""
    <div style="margin:6px 0;">
      <label style="color:{colors[i]}; font-weight:600; font-family:Arial; font-size:20px;">
        <input type="checkbox" checked
               onchange="toggleTrace({i}, this.checked)"
               style="margin-right:6px;">
        {col}
      </label>
    </div>
    """

widget_html = f"""
<div style="display:flex; align-items:flex-start;">
  <div>{chart_html}</div>
  <div style="border:1px solid #ddd; padding:15px; margin:40px 0 0 10px; width:300px; font-family:Arial;">
    <b>Select Drinks:</b>
    {checkbox_rows}
  </div>
</div>

<script>
function toggleTrace(index, isVisible) {{
  Plotly.restyle(document.getElementById('{chart_div_id}'), {{'visible': isVisible}}, [index]);
}}
</script>
"""

display(HTML(widget_html))

# **Findings**

## Seasonal Popularity by Drink

**Pumpkin Spice Latte — Fall (Sept–Nov)**
By far the most dramatic seasonal riser, spiking every single year like clockwork in September and dropping back to near-zero by December. This makes sense given ingredients and marketing: PSL is built on pumpkin spice syrup, cinnamon, nutmeg, and clove — flavors culturally coded as "fall comfort." Starbucks introduced the Pumpkin Spice Latte in 2003, and it went on to spur a multi-billion-dollar pumpkin spice industry spanning candles, popcorn, yogurt, and even car air fresheners — that broader cultural phenomenon is a big reason the search spike is so much taller than any other drink's seasonal bump: the term has become shorthand for "fall is here" beyond just the beverage itself.

**Refresher & Pink Drink — Spring/Summer (Apr–Aug)**
Both show moderate seasonal lift in the warmer months, tracking with actual product launches. Starbucks' 2026 summer menu arrived May 12 with a new Tropical Butterfly Refresher featuring guava and passionfruit flavors, and the brand followed up in June with a Blue Coconut Refresher and Iced Blue Coconut Matcha. This pattern of dropping a new Refresher variant every few weeks through spring and summer is exactly why the Refresher line shows sustained (not single-spike) elevation in warm months — there's always a new flavor to search for. The most popular Refreshers in the 2026 summer lineup — Strawberry Açaí, Mango Dragonfruit, Pink Drink, and Pineapple Passionfruit — perform well specifically because of their light, fruity, hydrating profile, which is a natural seasonal fit for hot weather.

**Matcha Tea — Mild upward drift, less sharply seasonal**
Matcha shows smaller recurring bumps but its overall shape is more of a steady multi-year climb than a sharp seasonal spike — consistent with matcha riding a broader, longer wellness/aesthetic trend rather than a single calendar season. Its earthy, slightly bitter flavor profile lends itself to both iced summer drinks and cozy lattes, which is likely why it doesn't show the extreme single-season concentration PSL does.

**Caramel Macchiato & Nitro Cold Brew — flat, non-seasonal "staples"**
Both sit in a narrow, low-volatility band year-round with no real spikes. Their ingredients explain this well: caramel and espresso, and plain cold-brewed coffee, aren't flavor-of-the-moment items — they're default orders that people search for/order consistently regardless of season.

## Most Popular Drink Currently

Based on the graph, **Refresher** posts the highest value in the entire dataset (~100) right at the very end of the series (2025–2026) — the tallest point on the whole chart. This lines up with real business data: Refreshers weren't an immediate hit when launched, but have since grown into a $2 billion platform, and Starbucks has been treating it as a flagship innovation line — the company describes Refreshers as "one of the fast-growing beverage platforms at Starbucks." The graph's late spike is consistent with the steady stream of limited-time Refresher variants (Tropical Butterfly, Blue Coconut, Mango Strawberry) that Starbucks has been releasing roughly every 4–6 weeks through 2026 — each launch appears to generate its own fresh search bump, stacking into an overall upward trend rather than a single seasonal spike.

## Least Popular Drink (Last 5 Years)

**Nitro Cold Brew Coffee** is the flattest, lowest line throughout the entire recent window — it never breaks out of a narrow low band, even as Refresher, Frappuccino, and PSL show growing volatility. This is a bit counterintuitive given actual sales data: Cold Brew (the category nitro belongs to) makes up 66% of beverage sales in U.S. coffeehouses today — meaning it's likely one of the best-*selling* drinks in absolute terms, but it doesn't generate much *search* interest. That's a meaningful distinction for your dashboard: cold brew is probably too established/habitual a purchase to drive search behavior (people don't Google something they already order automatically), whereas Refreshers and Frappuccino generate searches specifically *because* they're novel, visually shareable, and frequently re-flavored.

## Why This Pattern Makes Sense Overall

The common thread: **drinks that change (new flavors, new colors, new limited-time variants) generate search spikes; drinks that stay the same don't**, regardless of how popular they actually are in sales. PSL, Refreshers, and increasingly Frappuccino behave like "event" drinks tied to marketing calendars and social virality (color-changing drinks, celebrity promos, social-media-driven flavor drops), while Caramel Macchiato and Nitro Cold Brew behave like steady-state staples that people order out of habit rather than search for.